In [21]:
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pykeen.pipeline import pipeline

# Ingest and format the knowledge graph (kg.txt and ratings)

In [24]:
# Load Ratings
ratings = pd.read_csv('./ml-1m/ratings.csv', sep='::', engine='python', 
                      names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

# Treat all interactions as positive reward (any rating from 1-5) to optimize for engagement
interactions = ratings[['UserID', 'MovieID']].copy()

# Format as head, relation, tail
interactions['Head'] = 'user_' + interactions['UserID'].astype(str)
interactions['Relation'] = 'user.interact.movie'
interactions['Tail'] = 'movie_' + interactions['MovieID'].astype(str)

# load knowledge graph
kg = pd.read_csv('kg.txt', sep='\t', names=['Head', 'Relation', 'Tail'])

kg['Head'] = kg['Head'].astype(str)
kg['Tail'] = 'entity_' + kg['Tail'].astype(str)

interaction_t = interactions[['Head', 'Relation', 'Tail']]

graph = pd.concat([interaction_t, kg], ignore_index=True)

print(ratings)
print(kg)

         UserID  MovieID  Rating  Timestamp
0             1     1193       5  978300760
1             1      661       3  978302109
2             1      914       3  978301968
3             1     3408       4  978300275
4             1     2355       5  978824291
...         ...      ...     ...        ...
1000204    6040     1091       1  956716541
1000205    6040     1094       5  956704887
1000206    6040      562       5  956704746
1000207    6040     1096       4  956715648
1000208    6040     1097       4  956715569

[1000209 rows x 4 columns]
       Head            Relation         Tail
0       749    film.film.writer  entity_2347
1      1410  film.film.language  entity_2348
2      1037  film.film.language  entity_2348
3      1088    film.film.writer  entity_2349
4      1391  film.film.language  entity_2348
...     ...                 ...          ...
20777  2308    film.film.writer  entity_4284
20778   869  film.film.language  entity_2348
20779  1953     film.film.genre  entity

In [25]:
# add reverge edges
graph = pd.concat([graph, graph.rename(columns={'Head': 'Tail', 'Tail': 'Head'})], ignore_index=True)
# self loops
graph = pd.concat([graph, pd.DataFrame({'Head': graph['Head'], 'Relation': 'self_loop', 'Tail': graph['Head']})], ignore_index=True)

# Convert strings to ids for neural network
entities = pd.unique(graph[['Head', 'Tail']].values.ravel('K'))
entity_to_id = {entity: i for i, entity in enumerate(sorted(entities))}

relations = pd.unique(graph['Relation'])
relation_to_id = {relation: i for i, relation in enumerate(sorted(relations))}

graph['Head_ID'] = graph['Head'].map(entity_to_id)
graph['Tail_ID'] = graph['Tail'].map(entity_to_id)
graph['Relation_ID'] = graph['Relation'].map(relation_to_id)

numerical_graph = graph[['Head_ID', 'Relation_ID', 'Tail_ID']].values

# Write the mapped IDs to a tab-separated file for PyKEEN
graph[['Head_ID', 'Relation_ID', 'Tail_ID']].to_csv('train_triplets.txt', sep='\t', header=False, index=False)

print(graph)

                Head             Relation         Tail  Head_ID  Tail_ID  \
0             user_1  user.interact.movie   movie_1193    10714     7197   
1             user_1  user.interact.movie    movie_661    10714    10382   
2             user_1  user.interact.movie    movie_914    10714    10623   
3             user_1  user.interact.movie   movie_3408    10714     9511   
4             user_1  user.interact.movie   movie_2355    10714     8382   
...              ...                  ...          ...      ...      ...   
4083959  entity_4284            self_loop  entity_4284     4300     4300   
4083960  entity_2348            self_loop  entity_2348     2446     2446   
4083961  entity_2362            self_loop  entity_2362     2459     2459   
4083962  entity_5417            self_loop  entity_5417     5420     5420   
4083963  entity_2403            self_loop  entity_2403     2497     2497   

         Relation_ID  
0                  8  
1                  8  
2                 

In [ ]:
# Pretraining pipeline
result = pipeline(
    training='train_triplets.txt', # Path to numerical triplets
    testing='train_triplets.txt',  # Dummy testing to get it to run
    model='TransE',                # The translation model used in PGPR
    model_kwargs=dict(
        embedding_dim=100,         # Size specified in the PGPR paper
        scoring_fct_norm=1,        
    ),
    training_kwargs=dict(
        num_epochs=50,             # Paper suggests 50 epochs for training
        batch_size=64,             # Batch size from the paper
    ),
    optimizer='Adam',              # Adam optimizer from paper
    optimizer_kwargs=dict(
        lr=0.0001,                 # Learning rate from paper
    ),
    device='mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
)

# 2. Extract the trained weights
# These vectors are what the RL agent will use as its 'map'
entity_embeddings = result.model.entity_representations[0](indices=None).detach().cpu().numpy()
relation_embeddings = result.model.relation_representations[0](indices=None).detach().cpu().numpy()

# write
np.save('entity_embeddings.npy', entity_embeddings)
np.save('relation_embeddings.npy', relation_embeddings)


In [ ]:
kg_adj = []
for head, relation, tail in numerical_graph:
    kg_adj[head].append((relation, tail))
    
item_ids = set()

for entity_name, entity_id in entity_to_id.items():
    if entity_name.startswith('movie_'):
        item_ids.add(entity_id)

# Environment Setup, along with the Policy and Action Pruner

In [52]:
class Environment:
    def __init__(self, kg_adj, entity_embeddings, relation_embeddings, item_ids, max_steps=3):
        self.kg_adj = kg_adj
        self.entity_embeddings = entity_embeddings
        self.relation_embeddings = relation_embeddings
        self.item_ids = item_ids
        
        # Convert item_ids to a list and pre-slice their embeddings for fast vectorized operations
        self.item_ids_list = list(item_ids)
        self.item_embeddings_matrix = self.entity_embeddings[self.item_ids_list]
        
        self.max_steps = max_steps

        # State tracking variables
        self.user = None
        self.entity = None
        self.last_entity = None
        self.last_relation = None
        self.history = set()
        self.tstep = 0
        
        # Caching variable for normalized rewards
        self.max_user_score = 1.0 
        
    def get_state(self):
        return (self.user, self.entity, self.last_entity, self.last_relation)
        
    def get_valid_actions(self):
        valid_actions = []
        for relation, tail in self.kg_adj[self.entity]:
            if tail not in self.history and tail != self.user:
                valid_actions.append((relation, tail))
        return valid_actions

    def reset(self, user):
        self.user = user
        self.entity = user
        self.history = set()
        self.tstep = 0
        self.last_entity = user
        self.last_relation = 0 
        
        # --- NEW: Pre-calculate the max possible item score for this specific user ---
        u_emb = self.entity_embeddings[user]
        # Vectorized dot product against all movies at once
        all_item_scores = np.dot(self.item_embeddings_matrix, u_emb) 
        # Cache the highest score (use 1e-9 to prevent divide-by-zero if max score is negative/zero)
        self.max_user_score = max(np.max(all_item_scores), 1e-9)
        
        return self.get_state()
        
    def step(self, action):
        relation, next_entity = action
        
        self.history.add(self.entity)
        self.last_entity = self.entity
        self.last_relation = relation
        self.entity = next_entity
        self.tstep += 1
        
        done = self.tstep >= self.max_steps
        reward = 0.0
        
        if done:
            if self.entity in self.item_ids:
                u_emb = self.entity_embeddings[self.user]
                i_emb = self.entity_embeddings[self.entity]
                
                raw_score = np.dot(u_emb, i_emb)
                
                # --- NEW: Apply PGPR Equation 2 Normalization ---
                normalized_score = raw_score / self.max_user_score
                reward = max(0.0, float(normalized_score)) 
            else:
                reward = 0.0
                
        return self.get_state(), reward, done, {'valid_actions': self.get_valid_actions() if not done else []}

In [3]:
class ActionPruner:
    def __init__(self, entity_embeddings, relation_embeddings, max_actions=400):
        self.entity_embeddings = entity_embeddings
        self.relation_embeddings = relation_embeddings
        self.max_actions = max_actions

    def prune(self, user_id, valid_actions):
        """
        Implements User-Conditional Action Pruning (Eq. 1).
        Scores actions based on f((r,e)|u) = <u+r, e> and returns the top 'a' actions.
        """
        if not valid_actions:
            return []

        u_emb = self.entity_embeddings[user_id]
        action_scores = []

        for relation_id, tail_id in valid_actions:
            r_emb = self.relation_embeddings[relation_id]
            e_emb = self.entity_embeddings[tail_id]

            # 1-hop translational scoring function: <u + r, e>
            score = np.dot(u_emb + r_emb, e_emb)
            action_scores.append((score, (relation_id, tail_id)))

        # Sort by score descending and keep top 'max_actions'
        action_scores.sort(key=lambda x: x[0], reverse=True)
        pruned_actions = [act for score, act in action_scores[:self.max_actions]]

        return pruned_actions


class PolicyValueNetwork(nn.Module):
    def __init__(self, state_dim=400, action_space_size=400, hidden_dim1=512, hidden_dim2=256, dropout_rate=0.5):
        """
        Policy and Value networks sharing the same feature layers.
        State dim is 400 because it concatenates u, e_t, e_{t-1}, and r_t (4 * 100 dims).
        """
        super(PolicyValueNetwork, self).__init__()
        
        # Shared feature layers (W1 and W2)
        self.fc1 = nn.Linear(state_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        
        # Policy out (Wp) and Value out (Wv)
        self.policy_out = nn.Linear(hidden_dim2, action_space_size)
        self.value_out = nn.Linear(hidden_dim2, 1)
        
        self.dropout = nn.Dropout(dropout_rate)
        # The paper specifically uses the Exponential Linear Unit (ELU)
        self.elu = nn.ELU() 

    def forward(self, state, action_mask):
        """
        state: Tensor of shape (batch_size, state_dim)
        action_mask: Binary tensor of shape (batch_size, action_space_size) 
                     where 1 indicates a valid pruned action, 0 otherwise.
        """
        # x = dropout(sigma(dropout(sigma(s * W1)) * W2))
        x = self.elu(self.fc1(state))
        x = self.dropout(x)
        x = self.elu(self.fc2(x))
        x = self.dropout(x)

        # Policy Branch: pi(. | s, A_u)
        logits = self.policy_out(x)
        
        # Mask out invalid actions by setting their logits to a large negative number
        # This mathematically represents the Hadamard product masking before softmax
        masked_logits = logits.masked_fill(action_mask == 0, -1e9)
        action_probs = F.softmax(masked_logits, dim=-1)

        # Value Branch: v(s)
        state_value = self.value_out(x)

        return action_probs, state_value

# --- Example Initialization ---
# Assuming 'entity_embeddings' and 'relation_embeddings' are loaded from the pykeen output
# pruner = ActionPruner(entity_embeddings, relation_embeddings, max_actions=400)
# policy_net = PolicyValueNetwork(state_dim=400, action_space_size=400)

# Testing cells

In [17]:
# ==========================================
# 1. Generate Mock Data
# ==========================================
print("--- Initializing Mock Data ---")
num_entities = 1000
num_relations = 20
embed_dim = 100 

mock_entity_embeddings = np.random.randn(num_entities, embed_dim).astype(np.float32)
mock_relation_embeddings = np.random.randn(num_relations, embed_dim).astype(np.float32)
mock_item_ids = {100, 101, 102, 103, 104}

# Create a truly connected random adjacency list
mock_kg_adj = {i: [] for i in range(num_entities)}

for current_node in range(num_entities):
    # Give the user a huge out-degree to test pruning, give others a normal amount
    num_edges = 600 if current_node == 0 else np.random.randint(10, 50)
    
    for _ in range(num_edges):
        mock_kg_adj[current_node].append((
            np.random.randint(0, num_relations), 
            np.random.randint(0, num_entities)
        ))

# ==========================================
# 2. Instantiate Classes
# ==========================================
max_actions = 400

env = Environment(
    kg_adj=mock_kg_adj, 
    entity_embeddings=mock_entity_embeddings, 
    relation_embeddings=mock_relation_embeddings, 
    item_ids=mock_item_ids,
    max_steps=3
)

pruner = ActionPruner(
    entity_embeddings=mock_entity_embeddings, 
    relation_embeddings=mock_relation_embeddings, 
    max_actions=max_actions
)

policy_net = PolicyValueNetwork(
    state_dim=400, 
    action_space_size=max_actions
)

# ==========================================
# 3. Simulate One RL Step
# ==========================================
print("\n--- Starting RL Simulation Step ---")

# A. Reset Environment
state_tuple = env.reset(user=user_id)
print(f"1. Initial State Tuple (u, e_t, e_t-1, r_t): {state_tuple}")

# B. Get & Prune Actions
valid_actions = env.get_valid_actions()
print(f"2. Raw valid actions available: {len(valid_actions)}")

pruned_actions = pruner.prune(user_id=user_id, valid_actions=valid_actions)
print(f"3. Pruned actions count: {len(pruned_actions)} (Capped at {max_actions})")

# C. Prepare Network Inputs (State Vector & Mask)
# Extract individual IDs from state tuple
u, e_t, e_last, r_t = state_tuple

# Concatenate embeddings to form the 400-dim state vector
state_vec = np.concatenate([
    mock_entity_embeddings[u],
    mock_entity_embeddings[e_t],
    mock_entity_embeddings[e_last],
    mock_relation_embeddings[r_t] # r_t = 0 on reset, which acts as the dummy relation
])

# Convert to batch format for PyTorch: shape (1, 400)
state_tensor = torch.tensor(state_vec).unsqueeze(0)

# Create Action Mask: shape (1, 400)
# 1 for valid pruned actions, 0 for padding if pruned_actions < max_actions
action_mask = torch.zeros(1, max_actions)
action_mask[0, :len(pruned_actions)] = 1

# D. Forward Pass through Policy/Value Network
action_probs, state_value = policy_net(state_tensor, action_mask)

print("\n--- Network Outputs ---")
print(f"Action Probabilities Shape: {action_probs.shape}")
print(f"State Value Shape: {state_value.shape}")

# E. Sample an action and take a step
# Sample an index based on the probabilities
sampled_action_idx = torch.multinomial(action_probs, 1).item()
sampled_action = pruned_actions[sampled_action_idx]

print(f"\n--- Environment Transition ---")
print(f"Sampled Action Index: {sampled_action_idx}")
print(f"Executing Action (relation, tail): {sampled_action}")

next_state, reward, done, info = env.step(sampled_action)
print(f"Next State Tuple: {next_state}")
print(f"Reward: {reward}")
print(f"Done: {done}")

--- Initializing Mock Data ---

--- Starting RL Simulation Step ---
1. Initial State Tuple (u, e_t, e_t-1, r_t): (0, 0, 0, 0)
2. Raw valid actions available: 598
3. Pruned actions count: 400 (Capped at 400)

--- Network Outputs ---
Action Probabilities Shape: torch.Size([1, 400])
State Value Shape: torch.Size([1, 1])

--- Environment Transition ---
Sampled Action Index: 152
Executing Action (relation, tail): (2, 339)
Next State Tuple: (0, 339, 0, 2)
Reward: 0.0
Done: False


In [18]:
print("\n=== Starting Full Trajectory Simulation ===")

# 1. Reset Environment for a new episode
user_id = 0
state_tuple = env.reset(user=user_id)
print(f"Initial State (u, e_t, e_t-1, r_t): {state_tuple}")

# We will store the trajectory history here. 
# In REINFORCE, this is what we use to calculate the loss at the end of the episode.
trajectory_history = []
done = False
step = 1

while not done:
    print(f"\n--- Step {step} ---")
    
    # 2. Get valid actions from the environment
    valid_actions = env.get_valid_actions()
    
    # Handle Dead Ends: If a node has no outgoing edges (or all were visited)
    if not valid_actions:
        print(f"Reached a dead end at entity {state_tuple[1]}. Terminating early.")
        break
        
    # 3. Prune actions down to max_actions
    pruned_actions = pruner.prune(user_id=user_id, valid_actions=valid_actions)
    
    # 4. Prepare inputs for the neural network
    u, e_t, e_last, r_t = state_tuple
    state_vec = np.concatenate([
        mock_entity_embeddings[u],
        mock_entity_embeddings[e_t],
        mock_entity_embeddings[e_last],
        mock_relation_embeddings[r_t]
    ])
    state_tensor = torch.tensor(state_vec).unsqueeze(0) # Shape: (1, 400)
    
    action_mask = torch.zeros(1, pruner.max_actions)
    action_mask[0, :len(pruned_actions)] = 1
    
    # 5. Forward Pass
    action_probs, state_value = policy_net(state_tensor, action_mask)
    
    # 6. Sample an action from the probability distribution
    sampled_action_idx = torch.multinomial(action_probs, 1).item()
    sampled_action = pruned_actions[sampled_action_idx]
    
    print(f"Current Entity: {e_t}")
    print(f"Sampled Action (Relation, Next Entity): {sampled_action}")
    
    # 7. Take the step in the environment
    next_state, reward, done, _ = env.step(sampled_action)
    
    # Store step data for future RL training
    trajectory_history.append({
        'step': step,
        'state_tensor': state_tensor,
        'action_idx': sampled_action_idx,
        'action_prob': action_probs[0, sampled_action_idx].item(),
        'value': state_value.item(),
        'reward': reward
    })
    
    print(f"Transitioned to State: {next_state}")
    if done:
        print(f"\nTerminal Reward Reached: {reward}")
        if reward > 0:
            print("Success! The agent landed on a valid target item.")
        else:
            print("Failure. The agent did not land on a valid target item.")
        
    # Update state for the next iteration
    state_tuple = next_state
    step += 1

print("\n=== Trajectory Complete ===")
print(f"Total steps successfully executed: {len(trajectory_history)}")


=== Starting Full Trajectory Simulation ===
Initial State (u, e_t, e_t-1, r_t): (0, 0, 0, 0)

--- Step 1 ---
Current Entity: 0
Sampled Action (Relation, Next Entity): (11, 646)
Transitioned to State: (0, 646, 0, 11)

--- Step 2 ---
Current Entity: 646
Sampled Action (Relation, Next Entity): (5, 281)
Transitioned to State: (0, 281, 646, 5)

--- Step 3 ---
Current Entity: 281
Sampled Action (Relation, Next Entity): (1, 248)
Transitioned to State: (0, 248, 281, 1)

Terminal Reward Reached: 0.0
Failure. The agent did not land on a valid target item.

=== Trajectory Complete ===
Total steps successfully executed: 3


# Implementing REINFORCE w/ Baseline to train the agent

In [54]:
def train_pgpr(env, policy_net, pruner, entity_embeddings, relation_embeddings, valid_user_ids,
                       num_episodes=300000, batch_size=64, learning_rate=0.0001, gamma=0.99):
    """
    Trains the PGPR agent using REINFORCE with Baseline and Gradient Accumulation (Batching).
    """
    print(f"=== Starting Batched PGPR Training ({num_episodes} Episodes | Batch Size: {batch_size}) ===")

    optimizer = optim.Adam(policy_net.parameters(), lr=learning_rate)
    
    history_rewards = []
    history_losses = []
    
    # 1. Initialize gradients to zero BEFORE the loop starts
    optimizer.zero_grad()
    
    batch_loss = 0.0
    batch_reward = 0.0

    for episode in range(1, num_episodes + 1):
        # Reset Environment for a random user
        user_id = np.random.choice(valid_user_ids) 
        state_tuple = env.reset(user=user_id)
        
        trajectory = []
        done = False
        
        # --- Phase 1: Collect Trajectory ---
        while not done:
            valid_actions = env.get_valid_actions()
            if not valid_actions:
                break 
                
            pruned_actions = pruner.prune(user_id, valid_actions) 
            
            # Prepare state tensor
            u, e_t, e_last, r_t = state_tuple
            state_vec = np.concatenate([
                entity_embeddings[u],
                entity_embeddings[e_t],
                entity_embeddings[e_last],
                relation_embeddings[r_t]
            ])
            state_tensor = torch.tensor(state_vec).unsqueeze(0)
            
            # Prepare action mask
            action_mask = torch.zeros(1, pruner.max_actions)
            action_mask[0, :len(pruned_actions)] = 1
            
            # Forward pass
            action_probs, state_value = policy_net(state_tensor, action_mask)
            
            # Sample action
            m = torch.distributions.Categorical(probs=action_probs)
            sampled_action_idx = m.sample()
            sampled_action = pruned_actions[sampled_action_idx.item()]
            
            # Log probability
            log_prob = m.log_prob(sampled_action_idx)
            
            # Take environment step
            next_state, reward, done, _ = env.step(sampled_action)
            
            trajectory.append({
                'log_prob': log_prob,
                'value': state_value,
                'reward': reward
            })
            
            state_tuple = next_state

        # --- Phase 2: Compute Loss (Accumulate) ---
        if trajectory:
            returns = []
            G = 0
            for step_data in reversed(trajectory):
                G = step_data['reward'] + gamma * G
                returns.insert(0, G)
                
            returns = torch.tensor(returns)
            if len(returns) > 1:
                returns = (returns - returns.mean()) / (returns.std() + 1e-9)
                
            policy_loss = []
            value_loss = []
            
            for step_data, G in zip(trajectory, returns):
                advantage = G - step_data['value'].item()
                policy_loss.append(-step_data['log_prob'] * advantage)
                
                G_tensor = torch.tensor([G]).unsqueeze(0) 
                value_loss.append(F.mse_loss(step_data['value'], G_tensor))
                
            # Total Loss for this single trajectory
            trajectory_loss = torch.stack(policy_loss).sum() + torch.stack(value_loss).sum()
            
            # Divide by batch size so the gradients are averaged, not just summed!
            loss = trajectory_loss / batch_size
            
            # 2. Accumulate the gradients (do NOT step yet)
            loss.backward()
            
            batch_loss += trajectory_loss.item()
            batch_reward += sum([t['reward'] for t in trajectory])

        # --- Phase 3: Step the Optimizer (Once per Batch) ---
        if episode % batch_size == 0 or episode == num_episodes:
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=1.0)
            
            # 3. Take the step and reset gradients for the next batch
            optimizer.step()
            optimizer.zero_grad()
            
            # Logging
            history_rewards.append(batch_reward / batch_size)
            history_losses.append(batch_loss / batch_size)
            
            if (episode / batch_size) % 100 == 0:
                print(f"Episode {episode}/{num_episodes} | Avg Batch Loss: {history_losses[-1]:.4f} | Avg Batch Reward: {history_rewards[-1]:.4f}")
            
            # Reset batch trackers
            batch_loss = 0.0
            batch_reward = 0.0

    print("=== Training Complete ===")
    return history_rewards, history_losses

In [55]:
entity_embeddings = np.load('entity_embeddings.npy')
relation_embeddings = np.load('relation_embeddings.npy')

print(f"Loaded Entity Embeddings Shape: {entity_embeddings.shape}")
print(f"Loaded Relation Embeddings Shape: {relation_embeddings.shape}")

# 2. Extract your items (movies) and knowledge graph from the pykeen setup
# Assuming 'entity_to_id', 'relation_to_id', and 'numerical_graph' 
# are still available from your earlier data processing cell:

item_ids = set()
for entity_name, entity_id in entity_to_id.items():
    if entity_name.startswith('movie_'):
        item_ids.add(entity_id)

# Rebuild the adjacency list for the RL Environment
kg_adj = {i: [] for i in range(len(entity_to_id))}
for head, relation, tail in numerical_graph:
    kg_adj[head].append((relation, tail))

# 3. Instantiate your RL classes with the real data
env = Environment(
    kg_adj=kg_adj, 
    entity_embeddings=entity_embeddings, 
    relation_embeddings=relation_embeddings, 
    item_ids=item_ids,
    max_steps=3
)

pruner = ActionPruner(
    entity_embeddings=entity_embeddings, 
    relation_embeddings=relation_embeddings, 
    max_actions=400
)

policy_net = PolicyValueNetwork(
    state_dim=400, 
    action_space_size=400
)

valid_user_ids = []
for entity_name, entity_id in entity_to_id.items():
    if str(entity_name).startswith('user_'):
        valid_user_ids.append(entity_id)

# 4. Kick off the training!
rewards, losses = train_pgpr(env, policy_net, pruner, entity_embeddings, relation_embeddings, valid_user_ids)

Loaded Entity Embeddings Shape: (16754, 100)
Loaded Relation Embeddings Shape: (9, 100)
=== Starting Batched PGPR Training (300000 Episodes | Batch Size: 64) ===


KeyboardInterrupt: 

In [53]:
def policy_guided_path_reasoning(user_id, kg_adj, policy_net, pruner, entity_embeddings, relation_embeddings, item_ids, K_list=[25, 5, 1]):
    print(f"--- Starting Path Reasoning for User {id_to_entity.get(user_id, user_id)} ---")
    T = len(K_list)
    
    # --- NEW: Pre-calculate max score for the inference user ---
    item_ids_list = list(item_ids)
    item_embs = entity_embeddings[item_ids_list]
    u_emb = entity_embeddings[user_id]
    max_user_score = max(np.max(np.dot(item_embs, u_emb)), 1e-9)
    
    beam = [([user_id], 1.0, 0.0, (user_id, user_id, user_id, 0))]
    
    for t in range(T):
        K_t = K_list[t]
        next_beam = []
        
        for path, current_prob, current_reward, state_tuple in beam:
            u, e_t, e_last, r_t_val = state_tuple
            
            visited_entities = {path[0]} 
            for step in path[1:]:
                visited_entities.add(step[1])
            
            valid_actions = [(r, tail) for r, tail in kg_adj[e_t] if tail not in visited_entities]
            if not valid_actions:
                continue
                
            pruned_actions = pruner.prune(user_id=u, valid_actions=valid_actions)
            
            state_vec = np.concatenate([
                entity_embeddings[u],
                entity_embeddings[e_t],
                entity_embeddings[e_last],
                relation_embeddings[r_t_val]
            ])
            state_tensor = torch.tensor(state_vec).unsqueeze(0)
            
            action_mask = torch.zeros(1, pruner.max_actions)
            action_mask[0, :len(pruned_actions)] = 1
            
            with torch.no_grad():
                action_probs, _ = policy_net(state_tensor, action_mask)
            
            action_probs = action_probs[0].numpy()
            
            num_to_keep = min(K_t, len(pruned_actions))
            top_indices = np.argsort(action_probs[:len(pruned_actions)])[::-1][:num_to_keep]
            
            for idx in top_indices:
                action_p = action_probs[idx]
                r_next, e_next = pruned_actions[idx]
                
                new_path = path + [(r_next, e_next)]
                new_prob = current_prob * action_p
                
                # --- NEW: Apply Normalization to the step reward ---
                step_reward = 0.0
                if t == T - 1 and e_next in item_ids:
                    raw_score = float(np.dot(entity_embeddings[u], entity_embeddings[e_next]))
                    step_reward = max(0.0, raw_score / max_user_score)
                    
                new_reward = current_reward + step_reward
                new_state_tuple = (u, e_next, e_t, r_next)
                
                next_beam.append((new_path, new_prob, new_reward, new_state_tuple))
        
        beam = next_beam
        
    P_T, Q_T, R_T = [], [], []
    for path, prob, reward, _ in beam:
        final_entity = path[-1][1]
        if final_entity in item_ids:
            P_T.append(path)
            Q_T.append(prob)
            R_T.append(reward)
            
    return P_T, Q_T, R_T

In [51]:
# 1. Create reverse dictionaries so we can read the output
id_to_entity = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

# 2. Pick a test user (let's just grab the first valid user in your list)
test_user_id = valid_user_ids[0]
print(f"--- Running Inference for {id_to_entity[test_user_id]} ---")

# 3. Run Algorithm 1: Policy-Guided Path Reasoning
# Note: K_list=[25, 5, 1] means we explore the top 25 choices on step 1, 
# then 5 choices on step 2, and 1 choice on step 3.
P_T, Q_T, R_T = policy_guided_path_reasoning(
    user_id=test_user_id, 
    kg_adj=env.kg_adj, 
    policy_net=policy_net, 
    pruner=pruner, 
    entity_embeddings=entity_embeddings, 
    relation_embeddings=relation_embeddings, 
    item_ids=env.item_ids,
    K_list=[25, 5, 1] 
)

print(f"Beam Search found {len(P_T)} raw paths ending in a valid movie.")

# 4. Deduplicate and Extract the Best Explanations
# As per paper Section 3.4: "For each pair of (u, i_n)... select the path 
# with the highest generative probability... to interpret the reasoning process."
recommendations = {}
for path, prob, reward in zip(P_T, Q_T, R_T):
    final_item = path[-1][1]
    
    if final_item not in recommendations or prob > recommendations[final_item]['prob']:
        recommendations[final_item] = {
            'path': path,
            'prob': prob,
            'reward': reward
        }

# 5. Rank the Recommendations
# As per paper Section 3.4: "Rank the selected interpretable paths according to the path reward."
ranked_items = sorted(recommendations.items(), key=lambda x: x[1]['reward'], reverse=True)

print(f"\n=== Top Recommendations & Explanations ===")
for i, (item_id, data) in enumerate(ranked_items[:10]): # Show top 10
    item_name = id_to_entity[item_id]
    print(f"\n{i+1}. Recommend: {item_name}")
    print(f"   Reward Score: {data['reward']:.4f} | Path Probability: {data['prob']:.4f}")
    
    # Translate the path into a readable string
    path_str = id_to_entity[test_user_id]
    for relation_id, tail_id in data['path'][1:]:
        rel_name = id_to_relation[relation_id]
        tail_name = id_to_entity[tail_id]
        path_str += f" -> [{rel_name}] -> {tail_name}"
    
    print(f"   Reasoning: {path_str}")

--- Running Inference for user_1 ---
--- Starting Path Reasoning for User 10714 ---
Beam Search found 125 raw paths ending in a valid movie.

=== Top Recommendations & Explanations ===

1. Recommend: movie_2137
   Reward Score: 0.3676 | Path Probability: 0.0000
   Reasoning: user_1 -> [user.interact.movie] -> movie_1907 -> [user.interact.movie] -> user_392 -> [user.interact.movie] -> movie_2137

2. Recommend: movie_2491
   Reward Score: 0.3583 | Path Probability: 0.0000
   Reasoning: user_1 -> [user.interact.movie] -> movie_1907 -> [user.interact.movie] -> user_4784 -> [user.interact.movie] -> movie_2491

3. Recommend: movie_671
   Reward Score: 0.3524 | Path Probability: 0.0000
   Reasoning: user_1 -> [user.interact.movie] -> movie_1029 -> [user.interact.movie] -> user_1373 -> [user.interact.movie] -> movie_671

4. Recommend: movie_149
   Reward Score: 0.3502 | Path Probability: 0.0000
   Reasoning: user_1 -> [user.interact.movie] -> movie_608 -> [user.interact.movie] -> user_1845 -> 